# Feature Pool Construction

Consolidates the variable labels reported by 11 prior ICU-delirium studies into a
single candidate pool used as input to the LLM-based feature selection experiment.

**Input**  `features_from_papers.xlsx` — one column per source study, one raw label per row.

**Output**

| file | contents |
|---|---|
| `feature_pool.txt` | final candidate pool, one name per line |
| `variable_mapping.txt` | unified name ← original labels |
| `paper_feature_matrix.txt` | per-paper usage matrix |
| `excluded_features.txt` | excluded labels with reasons |

Run top to bottom. All consistency checks are enforced with `assert`.

In [ ]:
"""Literature-based feature pool construction for ICU delirium prediction.

Variable labels reported by 11 prior studies are consolidated into a single
candidate pool. Two rules govern the consolidation.

Rule 1 (Identity)
    Labels denoting the same underlying quantity are merged. This covers
    (a) notational synonyms  - WBC = White blood cell count
    (b) identical DB fields  - First care unit = Type of initial ICU admission
    Being a derived quantity is not grounds for merging: indices that the
    source studies reported as separate predictors with their own effect
    estimates (BMI, NLR, MBG, MAG, MAGE, ...) are kept distinct.

Rule 2 (Determinacy)
    A variable is excluded if its value is not determined for a single patient
    at 24 hours after ICU admission - either because the value is not yet
    fixed at that time point (temporal violation) or because it depends on
    fitting the whole cohort (independence violation).

Deliberately not applied
    Number of reporting papers, missingness rate, expected importance, and
    redundancy between variables. Judging redundancy (derived indices,
    superset-subset relations) is the task assigned to the LLM and is not
    pre-filtered here. No prior work in this line (DeLLiriuM, LLM-Select,
    Zhang 2023, Mei 2024) regulates redundancy either.

Out of scope
    Extraction artefacts that are not clinical variables (e.g. the 'Others'
    cell of a source table) are dropped before the rules are applied.

Outputs
    feature_pool.txt          final candidate pool, one name per line
    variable_mapping.txt      unified name <- original labels
    paper_feature_matrix.txt  per-paper usage matrix (tab separated)
    excluded_features.txt     excluded labels with reasons
"""

import pandas as pd
from collections import defaultdict

INPUT_PATH  = "features_from_papers.xlsx"
POOL_SIZE   = 107          # asserted below; update together with the declarations

df = pd.read_excel(INPUT_PATH)
PAPER_COLS = [c for c in df.columns if not str(c).startswith("Unnamed")]

paper_vars = {c: [v.strip() for v in df[c].dropna()] for c in PAPER_COLS}
all_vars   = sorted({v for vs in paper_vars.values() for v in vs}, key=str.lower)

print(f"source studies      : {len(PAPER_COLS)}")
print(f"distinct raw labels : {len(all_vars)}")

In [ ]:
# =============================================================================
# Pool declaration
#   feature(name, variants)   include in the pool; `name` is the label that
#                             appears in the paper's supplementary table
#   exclude(reason, variants) drop from the pool, recording the governing rule
#   The same raw label may be assigned to two features (1:N is allowed).
# =============================================================================
POOL     = {}   # unified name -> list of original labels
EXCLUDED = {}   # original label -> reason for exclusion

def feature(name, variants):
    POOL.setdefault(name, []).extend(variants)

def exclude(reason, variants):
    for v in variants:
        EXCLUDED[v] = reason


# --- Demographics ---
feature("Age", ["Age"])
feature("Sex", ["Gender", "Sex"])
feature("Ethnicity", ["Race", "Ethnicity"])
feature("Body Mass Index", ["BMI", "BMI (Body Mass Index)"])
feature("Height", ["Height"])
feature("Body Weight", ["Weight", "Body weight"])
feature("Marital Status", ["Marital status"])
feature("Insurance", ["Insurance"])

# --- Vital signs ---
feature("Heart Rate", ["Heart rate", "Heart rate (HR)"])
feature("Respiratory Rate", ["Respiratory rate", "Respiratory rate (RR)"])
feature("Temperature", ["Temperature", "Body temperature"])
feature("Oxygen Saturation", [
    "SpO₂", "SpO₂ (Oxygen saturation)", "SpO₂ (Oxyhemoglobin saturation)",
    "Oxygen saturation (SpO2)", "Oxygen saturation (SpO₂)",
])
feature("Systolic Blood Pressure", [
    "Systolic blood pressure", "Systolic blood pressure (SBP)", "SBP (Systolic BP)",
])
feature("Diastolic Blood Pressure", [
    "Diastolic blood pressure", "Diastolic blood pressure (DBP)", "DBP (Diastolic BP)",
])
feature("Mean Blood Pressure", [
    "Mean arterial pressure", "Mean arterial pressure (MAP)", "Mean blood pressure (MBP)",
])
# One source cell listed two variables together; assign it to both features.
feature("Systolic Blood Pressure", ["Arterial pressure (SBP, DBP)"])
feature("Diastolic Blood Pressure", ["Arterial pressure (SBP, DBP)"])

# --- Blood glucose ---
# Mei 2024 reported baseline glucose, MBG, MAG, MAGE, GLI and LAGE as separate
# predictors with individual odds ratios, so Rule 1 keeps them distinct.
feature("Glucose", ["Blood glucose", "Glucose", "Baseline blood glucose"])
feature("Mean Blood Glucose", ["Mean blood glucose (MBG)"])
feature("Mean Absolute Glucose", ["Mean absolute glucose (MAG)"])
feature("Mean Amplitude Of Glycemic Excursions", ["Mean amplitude of glycemic excursions (MAGE)"])
feature("Glycemic Lability Index", ["Glycemic lability index (GLI)"])
feature("Largest Amplitude Of Glycemic Excursions", [
    "Largest amplitude of glycemic excursions (LAGE",
])
exclude("[Rule 2] Independence violation - LGMM latent classes depend on fitting the whole cohort", [
    "MBG trajectory (latent growth mixture modeling, LGMM)",
])

# --- Laboratory: complete blood count ---
feature("White Blood Cell Count", [
    "WBC", "WBC (White blood cell count)", "WBC (White Blood Cell)", "WBC (White Blood Cells)",
    "White blood cell (WBC)", "White blood cell count", "White Blood Cell count (WBC)",
    "White blood cell count (WBC)", "White blood cells (WBC)",
])
feature("Hemoglobin", ["Hemoglobin"])
feature("Hematocrit", ["Hematocrit"])
feature("Platelet Count", ["Platelet", "Platelet count", "Platelets"])
feature("Red Blood Cell Count", ["Red blood cell count"])
feature("Red Cell Distribution Width", [
    "RDW (Red cell distribution width)", "Red cell distribution width (RDW)",
])
feature("Mean Corpuscular Volume", ["MCV (Mean corpuscular volume)"])
feature("Mean Corpuscular Hemoglobin", ["MCH (Mean corpuscular hemoglobin)"])
feature("Mean Corpuscular Hemoglobin Concentration", [
    "MCHC (Mean corpuscular hemoglobin concentration)",
])
feature("Neutrophil Count", ["Neutrophil count"])
feature("Lymphocyte Count", ["Lymphocyte count"])
feature("Monocyte Count", ["Monocyte count"])

# --- Laboratory: inflammatory ratios ---
feature("Neutrophil To Lymphocyte Ratio", ["NLR (Neutrophil-to-Lymphocyte Ratio)"])
feature("Platelet To Lymphocyte Ratio", ["PLR (Platelet-to-Lymphocyte Ratio)"])
feature("Lymphocyte To Monocyte Ratio", ["LMR (Lymphocyte-to-Monocyte Ratio)"])

# --- Laboratory: chemistry ---
feature("Blood Urea Nitrogen", [
    "BUN", "BUN (Blood urea nitrogen)", "BUN (Blood Urea Nitrogen)", "Blood urea nitrogen (BUN)",
])
feature("Creatinine", [
    "Creatinine", "Creatinine (Cr)", "Blood creatinine", "Serum creatinine",
    "serum creatinine (Scr) during hospitalization",
])
feature("Blood Sodium", ["Sodium", "Sodium (Na)", "Blood sodium"])
feature("Potassium", ["Potassium", "Potassium (K)", "Blood potassium"])
feature("Chloride", ["Chloride"])
feature("Calcium", ["Calcium", "Calcium (total)"])
feature("Magnesium", ["Magnesium"])
feature("Phosphate", ["Phosphate"])
feature("Bicarbonate", ["Bicarbonate", "HCO₃ (Bicarbonate)"])
feature("Anion Gap", ["Anion gap", "Anion Gap (AG)"])
feature("Albumin", ["Albumin", "Serum albumin"])
feature("Bilirubin", ["Bilirubin", "Total bilirubin", "TBIL (Total bilirubin)"])
feature("Alanine Aminotransferase", [
    "ALT", "Alanine aminotransferase (ALT)", "Alanine transaminase (ALT)",
])
feature("Aspartate Aminotransferase", [
    "AST", "Aspartate aminotransferase (AST)", "Aspartate transaminase (AST)",
])
feature("Alkaline Phosphatase", ["Alkaline phosphatase (ALP)"])
feature("Lactate", ["Lactate"])
feature("Glomerular Filtration Rate", [
    "Estimated glomerular filtration rate (eGFR)", "GFR (Glomerular Filtration Rate)",
])

# --- Laboratory: arterial blood gas ---
feature("Blood pH", ["pH"])
feature("Arterial Partial Pressure Of Oxygen", [
    "PO₂", "pO₂", "PO2 (Partial pressure of oxygen)", "Partial pressure of oxygen (PaO₂)",
])
feature("Arterial Partial Pressure Of Carbon Dioxide", [
    "PCO₂", "pCO₂", "PCO2 (Partial pressure of carbon dioxide)",
    "Partial pressure of carbon dioxide (PaCO₂)",
])
feature("Base Excess", ["BE (Base Excess)"])
feature("Total Carbon Dioxide", ["Total CO₂"])

# --- Laboratory: coagulation ---
feature("Prothrombin Time", [
    "PT (Prothrombin time)", "PT (Prothrombin Time)", "Prothrombin time (PT)",
])
feature("Partial Thromboplastin Time", [
    "PTT", "PTT (Partial thromboplastin time)", "PTT (Partial Thromboplastin Time)",
    "Partial thromboplastin time (PTT)", "Activated partial thromboplastin time (aPTT)",
])
feature("International Normalized Ratio", ["INR", "International normalized ratio (INR)"])

# --- Urine ---
feature("Urine Output", ["Urine output"])

# --- Comorbidities ---
feature("Hypertension", ["Hypertension"])
feature("Diabetes", [
    "Diabetes", "Diabetes mellitus", "Diabetes mellitus (DM)",
    "Diabetes mellitus with chronic comorbidities",
])
feature("Acute Myocardial Infarction", [
    "Myocardial infarction", "Myocardial infarction (MI)", "Acute myocardial infarction (AMI)",
])
feature("Heart Failure", [
    "Congestive heart failure", "Congestive heart failure (CHF)", "Heart failure",
])
feature("Peripheral Vascular Disease", ["Peripheral vascular disease"])
feature("Cardiovascular Disease", ["Cardiovascular disease (CVD)"])
feature("Cerebrovascular Disease", ["Cerebrovascular disease"])
feature("Stroke", ["Stroke"])
feature("Chronic Obstructive Pulmonary Disease", [
    "Chronic pulmonary disease", "Chronic lung disease",
    "Chronic obstructive pulmonary disease (COPD)", "COPD",
])
feature("Renal Disease", [
    "Chronic kidney disease (CKD)", "Chronic kidney dysfunction (CKD)",
    "Chronic renal disease (CKD)", "CKD (Chronic Kidney Disease)", "Kidney disease",
    "Renal disease",
])
feature("Liver Disease", [
    "Chronic liver disease (CLD)", "Chronic hepatitis", "Liver disease", "Severe liver disease",
])
feature("Cancer", ["Cancer", "Malignancy (Cancer)", "Malignant neoplasm"])
feature("Dementia", ["Dementia"])
feature("Depression", ["Depression"])
feature("Anemia", ["Anemia"])
feature("Sepsis", ["Sepsis"])
feature("Alcohol Abuse", ["Alcohol abuse"])
feature("Nicotine Dependence", ["Nicotine dependence"])

# --- Severity scores ---
feature("Sequential Organ Failure Assessment Score", [
    "SOFA", "SOFA score", "SOFA (Sequential Organ Failure Assessment)",
    "Sequential Organ Failure Assessment (SOFA)",
])
feature("Simplified Acute Physiology Score II", [
    "SAPS II", "SAPS II (Simplified Acute Physiology Score II)",
])
feature("Acute Physiology and Chronic Health Evaluation (APACHE) Score", [
    "APS III (Acute Physiology Score III)", "APSIII (Acute Physiology Score III)",
    "Acute physiology score", "Acute Physiology Score III (APS-III)",
])
feature("Oxford Acute Severity Of Illness Score", [
    "OASIS (Oxford Acute Severity of Illness Score)",
])
feature("Logistic Organ Dysfunction Score", ["LODS", "LODS (Logistic Organ Dysfunction Score)"])
feature("Systemic Inflammatory Response Syndrome", [
    "SIRS (Systemic Inflammatory Response Syndrome)",
])
feature("Glasgow Coma Scale", ["GCS (Glasgow Coma Scale)", "Glasgow Coma Scale (GCS)"])
feature("Acute Physiology and Chronic Health Evaluation (APACHE) Score", ["APACHE II score"])
feature("Charlson Comorbidity Index", [
    "Charlson Comorbidity Index", "CCI (Charlson Comorbidity Index)", "Comorbidities",
])

# --- Nutritional indices ---
feature("Prognostic Nutritional Index", ["PNI (Prognostic Nutritional Index)"])
feature("Geriatric Nutritional Risk Index", ["GNRI (Geriatric Nutritional Risk Index)"])

# --- Treatments and interventions ---
feature("Mechanical Ventilation", [
    "Mechanical ventilation", "Mechanical ventilation status", "Mechanical ventilation use",
    "MV (Mechanical Ventilation)",
])
exclude("[Rule 2] Temporal violation - ventilation duration is undetermined at 24 h and is lengthened by delirium", [
    "Ventilator service time in ICU (days)",
])
feature("Vasopressor Use", ["Vasopressor use"])
feature("Renal Replacement Therapy", [
    "Renal replacement therapy (RRT)", "CRRT (Continuous Renal Replacement Therapy)",
])
feature("Sedation Use", ["Sedation use", "Sedative drug use", "Sedative use", "Use of propofol"])
feature("Use Of Midazolam", ["Use of midazolam", "Benzodiazepine drugs"])
feature("Statin Drugs", ["Statin drugs"])
feature("Antibiotic Drug Use", ["Antibiotic drug use"])
feature("Acute Kidney Injury", ["AKI (Acute Kidney Injury)"])

# --- ICU and hospital ---
# Rule 1(b): the MIMIC-IV documentation defines first_careunit as the ICU type.
feature("Type Of Initial ICU Admission", [
    "Type of initial ICU admission (ICU type)", "First care unit",
])
exclude("[Rule 2] Temporal violation - undetermined at 24 h and lengthened by delirium", [
    "ICU length of stay", "ICU length of stay (days)", "Total length of stay in ICU (ICU LOS)",
])
exclude("[Rule 2] Temporal violation - undetermined at 24 h and lengthened by delirium", ["Hospital length of stay"])

# --- Surgery ---
feature("Elective Surgery", ["Elective surgery"])
feature("Coronary Artery Bypass Grafting", ["CABG (Coronary artery bypass grafting)"])
feature("Aortic Surgery", ["Aortic surgery"])
feature("Heart Valve Surgery", ["Heart valve surgery"])

# --- Infection ---
feature("Infection", ["Infection"])
feature("Gram Negative", ["Gram negative"])
feature("Gram Positive", ["Gram positive"])
feature("Type Of Microorganism", ["Type of microorganism"])

# --- Out of scope ---
exclude("[Out of scope] The 'Others' cell of a source table; an extraction artefact, not a clinical variable", ["Others"])

In [ ]:
# =============================================================================
# Consistency checks
#   Every raw label in the source table must be either included or excluded,
#   never both, and no declared label may be absent from the source table.
# =============================================================================
var2feats = defaultdict(list)
for name, variants in POOL.items():
    for v in variants:
        var2feats[v].append(name)

declared = set(var2feats) | set(EXCLUDED)
missing  = [v for v in all_vars if v not in declared]
overlap  = sorted(set(var2feats) & set(EXCLUDED))
unused   = sorted(declared - set(all_vars))

assert not missing, f"{len(missing)} raw labels were never declared: {missing}"
assert not overlap, f"declared as both included and excluded: {overlap}"
assert not unused,  f"declared labels absent from the source table (typo?): {unused}"

feature_pool = sorted(POOL, key=str.lower)
assert len(feature_pool) == POOL_SIZE, \
    f"pool size is {len(feature_pool)}, expected {POOL_SIZE}"

print(f"raw labels excluded : {len(EXCLUDED)}")
print(f"final pool size     : {len(feature_pool)}")

In [ ]:
# =============================================================================
# Per-paper usage and file output
# =============================================================================
paper_feats = {c: {f for v in vs for f in var2feats.get(v, [])}
               for c, vs in paper_vars.items()}

with open("feature_pool.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(feature_pool)) + "\n")

with open("variable_mapping.txt", "w", encoding="utf-8") as f:
    f.write("Unified Feature Name\tOriginal Variable Names\tVariant Count\n")
    for name in sorted(feature_pool):
        ov = sorted(set(POOL[name]))
        f.write(f"{name}\t{' | '.join(ov)}\t{len(ov)}\n")

with open("paper_feature_matrix.txt", "w", encoding="utf-8") as f:
    f.write("Feature\t" + "\t".join(PAPER_COLS) + "\tPaper Count\n")
    rows = [(feat, [int(feat in paper_feats[c]) for c in PAPER_COLS])
            for feat in feature_pool]
    rows.sort(key=lambda r: (-sum(r[1]), r[0].lower()))
    for feat, flags in rows:
        f.write(f"{feat}\t" + "\t".join(map(str, flags)) + f"\t{sum(flags)}\n")

with open("excluded_features.txt", "w", encoding="utf-8") as f:
    f.write("Original Variable Name\tReason for Exclusion\n")
    for v in sorted(EXCLUDED, key=str.lower):
        f.write(f"{v}\t{EXCLUDED[v]}\n")

print("written: feature_pool.txt, variable_mapping.txt, "
      "paper_feature_matrix.txt, excluded_features.txt")